# Pipeline walkthrough

This notebook runs the same cleaning pipeline as `clean.py`, **one step at a time**, so you can see what each step does to the data.

The logic lives in `cleaning/*.py`. The notebook only calls it and shows the results, so the script and the notebook can never disagree.

Run `python scripts/generate_messy_data.py --out data/raw` once before this notebook.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# Find the project folder (the one that contains `cleaning/`), wherever Jupyter started
ROOT = Path.cwd()
while not (ROOT / "cleaning").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "cleaning").is_dir():
    raise FileNotFoundError(f"Can't find the project from {Path.cwd()}. "
                            "Put this notebook inside the project's notebooks/ folder.")
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

from cleaning import normalize as nz
from cleaning import pipeline

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

Project folder: d:\Mark imp\sales-data-cleanup\sales-data-cleanup


## Step 1: Load every export as text

Everything is read as text (`dtype=str`) so phone numbers keep their leading 0. Column names from the 3 systems are mapped to one schema.

In [2]:
raw, file_info = pipeline.load_exports(ROOT / "data" / "raw")
print(f"{len(raw):,} rows loaded")
pd.DataFrame(file_info)

4,698 rows loaded


,file,rows,columns
0,orders_2026_01.csv,713,11
1,orders_2026_02.csv,847,11
2,orders_2026_03.xlsx,777,11
3,orders_2026_04.xlsx,792,11
4,orders_2026_05.csv,754,12
5,orders_2026_06.csv,815,12


In [3]:
# Same column, three different names in the original files
for f in sorted((ROOT / "data" / "raw").iterdir()):
    df = pd.read_excel(f, nrows=0) if f.suffix == ".xlsx" else pd.read_csv(f, nrows=0)
    print(f"{f.name:22} {list(df.columns)[:4]}")

orders_2026_01.csv     ['Order ID', 'Order Date', 'Customer Name', 'Phone']
orders_2026_02.csv     ['Order ID', 'Order Date', 'Customer Name', 'Phone']
orders_2026_03.xlsx    ['order_no', 'date', 'customer', 'mobile']
orders_2026_04.xlsx    ['order_no', 'date', 'customer', 'mobile']
orders_2026_05.csv     ['OrderID', 'Date', 'Client', 'Phone Number']
orders_2026_06.csv     ['OrderID', 'Date', 'Client', 'Phone Number']


In [4]:
raw.sample(5, random_state=1)

,order_id,order_date,customer_name,phone,email,city,product,quantity,unit_price,payment_method,status,source_file,file_order,source_row,discount
3291,ORD-13237,"May 12, 2026",Tarek Soliman,+20 159-371-7532,TAREK.SOLIMAN63@GMAIL.COM,cairo,Wall Clock,3,410.0,Wallet,delivered,orders_2026_05.csv,4,164,
4221,ORD-13668,"Jun 10, 2026",Sherif Gamal,01528754377,sherif.gamal70@gmail.com,TANTA,Cotton Bed Sheet Set,1,1400.0,COD,Delivered,orders_2026_06.csv,5,340,0
2872,12168,20/04/2026,Kirollos Ezzat,1557405358,kirollos.ezzat66@yahoo.com,Cairo City,Ceramic Dinner Set (24 pcs),1,"2,300.00 EGP",COD,Delivered,orders_2026_04.xlsx,3,537,NaN
4041,ord13288,"May 28, 2026",HANA FAWZY,+20 129-419-6727,hana.fawzy88@outlook.com,Cairo City,Electric Kettle 1.7L,1,1250.0,Card,Returned,orders_2026_06.csv,5,160,-20
3591,ORD 13155,"May 01, 2026",Laila Mahmoud,01077049595,,El Mansoura,Cotton Bed Sheet Set,2,1400.0,cash,Completed,orders_2026_05.csv,4,464,


## Step 2: Drop fully empty rows

In [5]:
raw, empty_rows = pipeline.drop_empty_rows(raw)
print(f"dropped {empty_rows} empty rows, {len(raw):,} left")

dropped 43 empty rows, 4,655 left


## Step 3: Clean one value at a time

Every column has a small function in `cleaning/normalize.py`. Try them on single values first.

In [6]:
phones = ["01012345678", "00201012345678", "010 1234 5678", "+20 101-234-5678", "1012345678", "12345", "N/A"]
pd.DataFrame({"raw": phones, "clean": [nz.phone(p) for p in phones]})

,raw,clean
0,01012345678,+201012345678
1,00201012345678,+201012345678
2,010 1234 5678,+201012345678
3,+20 101-234-5678,+201012345678
4,1012345678,+201012345678
5,12345,NaN
6,N/A,NaN


In [7]:
samples = {
    "order_id": ["ord10001", "ORD 10001", "10001"],
    "order_date": ["2026-03-09", "09/03/2026", "Mar 09, 2026"],
    "customer_name": ["  mostafa   ADEL ", "MOSTAFA ADEL", "Mostafa Adel"],
    "city": ["Al Qahirah", " CAIRO ", "Cairo City"],
    "quantity": ["2 pcs", "0", "-2"],
    "unit_price": ["2,300.00 EGP", "EGP 2,400", ""],
}
rows = []
for col, values in samples.items():
    fn = pipeline.CLEANERS[col]
    rows += [{"column": col, "raw": v, "clean": fn(v)} for v in values]
pd.DataFrame(rows)

,column,raw,clean
0,order_id,ord10001,ORD-10001
1,order_id,ORD 10001,ORD-10001
2,order_id,10001,ORD-10001
3,order_date,2026-03-09,2026-03-09
4,order_date,09/03/2026,2026-03-09
5,order_date,"Mar 09, 2026",2026-03-09
6,customer_name,mostafa ADEL,Mostafa Adel
7,customer_name,MOSTAFA ADEL,Mostafa Adel
8,customer_name,Mostafa Adel,Mostafa Adel
9,city,Al Qahirah,Cairo


Now apply every function to every row. `changes` counts, per column, how many values were standardized, unusable or missing.

In [8]:
clean, changes = pipeline.standardize(raw)
pd.DataFrame(changes).T

,standardized,unusable,missing
order_id,3513,0,0
order_date,3110,0,0
customer_name,1928,0,0
phone,3777,29,86
email,575,129,249
city,3436,0,0
product,0,0,0
quantity,464,89,0
unit_price,2006,0,48
payment_method,4655,0,0


In [9]:
# The same 5 rows, before and after
cols = ["order_id", "order_date", "customer_name", "phone", "city", "quantity", "unit_price", "status"]
idx = raw.sample(5, random_state=7).index
pd.concat({"before": raw.loc[idx, cols], "after": clean.loc[idx, cols]}, axis=1)

before                                                                                                after                                                                                  
       order_id    order_date customer_name             phone         city quantity unit_price     status   order_id  order_date customer_name          phone      city quantity unit_price     status
3283   ord12923  May 31, 2026   Reem Girgis       01519990477         Giza        1     1150.0   returned  ORD-12923  2026-05-31   Reem Girgis  +201519990477      Giza        1     1150.0   returned
567   ORD 10170    2026-01-12  HANA SOLIMAN    00201556240084        TANTA    1 pcs      420.0  DELIVERED  ORD-10170  2026-01-12  Hana Soliman  +201556240084     Tanta        1      420.0  delivered
4696   ord13052  May 27, 2026    REEM SAMIR  +20 118-842-9450        Tanta        1     1100.0  Completed  ORD-13052  2026-05-27    Reem Samir  +201188429450     Tanta        1     1100.0  delivered
3501      12820  May 15, 2026    Hassan Ali    00201156348030  El Mansoura        2    EGP 650    Pending  ORD-12820  2026-05-15    Hassan Ali  +201156348030  Mansoura        2      650.0    pending
196       10612    2026-01-16  Fady Mansour     +201297627170      El Giza        1      490.0  Completed  ORD-10612  2026-01-16  Fady Mansour  +201297627170      Giza        1      490.0  delivered

## Step 4: Reject rows missing a required field

Nothing is guessed. A row without a usable quantity or price goes to the rejected log with its reason.

In [10]:
valid, rejected = pipeline.split_valid(clean, raw)
print(f"{len(valid):,} valid rows, {len(rejected):,} rejected")
rejected["reject_reason"].value_counts()

4,522 valid rows, 133 rejected


reject_reason
invalid or missing quantity                                   85
invalid or missing unit_price                                 44
invalid or missing quantity; invalid or missing unit_price     4
Name: count, dtype: int64

In [11]:
rejected[["source_file", "source_row", "order_id", "quantity", "unit_price", "reject_reason"]].head()

,source_file,source_row,order_id,quantity,unit_price,reject_reason
9,orders_2026_01.csv,11,10233,-2,2400.0,invalid or missing quantity
49,orders_2026_01.csv,51,ord10464,0,365.0,invalid or missing quantity
53,orders_2026_01.csv,55,ORD-10305,1,,invalid or missing unit_price
64,orders_2026_01.csv,66,ord10417,-3,1450.0,invalid or missing quantity
96,orders_2026_01.csv,98,ord10532,0,360.0,invalid or missing quantity


## Step 5: Remove duplicates

First repeats inside the same file, then the same order exported again in a later file. The copy from the **latest file** is kept, because it has the newest status.

In [12]:
final, dupes = pipeline.deduplicate(valid)
print(dupes)
print(f"{len(final):,} unique orders")

{'within_file': 131, 'across_files': 536, 'conflicting_orders': 31}
3,855 unique orders


In [13]:
# One order whose status changed between exports
changed = valid.groupby("order_id")["status"].nunique()
oid = changed[changed > 1].index[0]
print("raw copies:")
display(valid.loc[valid["order_id"] == oid, ["order_id", "source_file", "status"]])
print("kept:")
final.loc[final["order_id"] == oid, ["order_id", "source_file", "status"]]

raw copies:


,order_id,source_file,status
317,ORD-10101,orders_2026_01.csv,pending
948,ORD-10101,orders_2026_02.csv,delivered


kept:


,order_id,source_file,status
948,ORD-10101,orders_2026_02.csv,delivered


## Step 6: Build the outputs

Customers are grouped by phone number, because the same person's name is spelled several ways.

In [14]:
orders = pipeline.build_orders(final)
customers, merged = pipeline.build_customers(orders)
print(f"{len(orders):,} orders, {len(customers):,} customers, {merged} customers had more than one name spelling")
orders.head()

3,855 orders, 350 customers, 70 customers had more than one name spelling


,order_id,order_date,customer_name,phone,email,city,product,quantity,unit_price,discount,line_total,payment_method,status,source_file
0,ORD-10079,2026-01-01,Karim Ramadan,+201163280554,karim.ramadan4@outlook.com,Tanta,Memory Foam Pillow,2,650.0,0.0,1300.0,card,pending,orders_2026_01.csv
1,ORD-10126,2026-01-01,Habiba Kamal,+201562336371,habiba.kamal27@yahoo.com,Tanta,Memory Foam Pillow,1,650.0,0.0,650.0,mobile_wallet,delivered,orders_2026_01.csv
2,ORD-10159,2026-01-01,Nada Mansour,+201555177213,NaN,Cairo,Memory Foam Pillow,1,650.0,0.0,650.0,cash_on_delivery,cancelled,orders_2026_01.csv
3,ORD-10211,2026-01-01,Fatma Mahmoud,+201510106509,fatma.mahmoud19@gmail.com,Mansoura,Bath Towel Pack (4),2,780.0,0.0,1560.0,card,delivered,orders_2026_01.csv
4,ORD-10266,2026-01-01,Yasmin Fathy,+201085726382,yasmin.fathy38@yahoo.com,Tanta,Bath Towel Pack (4),1,880.0,0.0,880.0,cash_on_delivery,delivered,orders_2026_01.csv


In [15]:
# One customer, several spellings, one phone
phone = customers.loc[customers["name_spellings"] > 1, "phone"].iloc[0]
raw_names = raw.loc[clean["phone"] == phone, "customer_name"].unique()
print("spellings in the raw files:", list(raw_names))
customers[customers["phone"] == phone]

spellings in the raw files: ['Amr Kamal', 'AMRO KAMAL', '  Amr   Kamal ', 'amr kamal', '  Amro   Kamal ', 'AMR KAMAL', 'amro kamal', 'Amro Kamal']


,phone,customer_name,name_spellings,email,city,orders,delivered_revenue_egp,first_order,last_order
5,+201110043277,Amr Kamal,2,amr.kamal61@yahoo.com,Alexandria,12,26285.0,2026-01-03,2026-06-21


## Check: every row is accounted for

In [16]:
flow = {
    "raw rows": len(raw) + empty_rows,
    "empty": -empty_rows,
    "rejected": -len(rejected),
    "duplicates inside a file": -dupes["within_file"],
    "re-exported in a later file": -dupes["across_files"],
}
flow["clean orders"] = sum(flow.values())
assert flow["clean orders"] == len(orders)
pd.Series(flow, name="rows").to_frame()

,rows
raw rows,4698
empty,-43
rejected,-133
duplicates inside a file,-131
re-exported in a later file,-536
clean orders,3855
